## Data cleaning file 

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/middle_east.csv")

print(df.head())

               WEEK       REGION  COUNTRY   ADMIN1  \
0  06-February-2016  Middle East  Bahrain  Capital   
1  28-February-2026  Middle East  Bahrain  Capital   
2     07-March-2026  Middle East  Bahrain  Capital   
3     14-March-2026  Middle East  Bahrain  Capital   
4     21-March-2026  Middle East  Bahrain  Capital   

                   EVENT_TYPE    SUB_EVENT_TYPE  EVENTS  FATALITIES  \
0                     Battles       Armed clash       1           0   
1  Explosions/Remote violence  Air/drone strike       9           0   
2  Explosions/Remote violence  Air/drone strike       7           1   
3  Explosions/Remote violence  Air/drone strike       1           0   
4  Explosions/Remote violence  Air/drone strike       1           0   

   POPULATION_EXPOSURE       DISORDER_TYPE   ID  CENTROID_LATITUDE  \
0              69821.0  Political violence  285            26.1927   
1             126395.0  Political violence  285            26.1927   
2              57503.0  Political viol

In [3]:
print("Number of rows and columns in the datasets")
print()
print("Dataset Shape",df.shape)
df.isnull().sum()
# df.isnull().any().any()

Number of rows and columns in the datasets

Dataset Shape (155295, 13)


WEEK                       0
REGION                     0
COUNTRY                    0
ADMIN1                     0
EVENT_TYPE                 0
SUB_EVENT_TYPE             0
EVENTS                     0
FATALITIES                 0
POPULATION_EXPOSURE    33559
DISORDER_TYPE              0
ID                         0
CENTROID_LATITUDE          0
CENTROID_LONGITUDE         0
dtype: int64

In [4]:
missing_counts = df.isnull().sum()
missing_counts[missing_counts > 0]

POPULATION_EXPOSURE    33559
dtype: int64

In [5]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155295 entries, 0 to 155294
Data columns (total 13 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   WEEK                 155295 non-null  object 
 1   REGION               155295 non-null  object 
 2   COUNTRY              155295 non-null  object 
 3   ADMIN1               155295 non-null  object 
 4   EVENT_TYPE           155295 non-null  object 
 5   SUB_EVENT_TYPE       155295 non-null  object 
 6   EVENTS               155295 non-null  int64  
 7   FATALITIES           155295 non-null  int64  
 8   POPULATION_EXPOSURE  121736 non-null  float64
 9   DISORDER_TYPE        155295 non-null  object 
 10  ID                   155295 non-null  int64  
 11  CENTROID_LATITUDE    155295 non-null  float64
 12  CENTROID_LONGITUDE   155295 non-null  float64
dtypes: float64(3), int64(3), object(7)
memory usage: 15.4+ MB


,EVENTS,FATALITIES,POPULATION_EXPOSURE,ID,CENTROID_LATITUDE,CENTROID_LONGITUDE
count,155295.000000,155295.000000,1.217360e+05,155295.000000,155295.000000,155295.000000
mean,4.116778,3.627490,7.917423e+04,2602.789163,30.212089,41.328587
std,9.746724,34.105346,1.820226e+05,1009.762522,8.706671,6.281700
min,1.000000,0.000000,1.000000e+00,285.000000,12.508100,26.695700
25%,1.000000,0.000000,1.183950e+04,1375.000000,28.272800,36.190100
50%,2.000000,0.000000,3.115900e+04,3183.000000,33.283200,42.349600
75%,3.000000,1.000000,7.833275e+04,3410.000000,36.026500,44.788000
max,324.000000,8106.000000,1.261539e+07,4203.000000,43.496500,70.473400


In [6]:
# Fills missing POpulation using the median of that specific sub-region (ADMIN1)
df['POPULATION_EXPOSURE'] =df.groupby(['COUNTRY', 'ADMIN1'])['POPULATION_EXPOSURE'].transform(lambda x: x.fillna(x.median()))

# Secondary fallback: If an entire ADMIN1 region is missing numbers, fill using the broader country median
df['POPULATION_EXPOSURE'] == df.groupby('COUNTRY')['POPULATION_EXPOSURE'].transform(lambda x: x.fillna(x.median()))


c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keep

0         True
1         True
2         True
3         True
4         True
          ... 
155290    True
155291    True
155292    True
155293    True
155294    True
Name: POPULATION_EXPOSURE, Length: 155295, dtype: bool

In [7]:
print(df.columns.tolist())




['WEEK', 'REGION', 'COUNTRY', 'ADMIN1', 'EVENT_TYPE', 'SUB_EVENT_TYPE', 'EVENTS', 'FATALITIES', 'POPULATION_EXPOSURE', 'DISORDER_TYPE', 'ID', 'CENTROID_LATITUDE', 'CENTROID_LONGITUDE']


In [8]:
# Group by country using the exact column name
missing_by_country = df[df['POPULATION_EXPOSURE'].isnull()].groupby('COUNTRY').size()

# Sort the results in descending order
missing_by_country = missing_by_country.sort_values(ascending=False)
missing_by_country


COUNTRY
Indian Ocean            487
Mediterranean Sea       276
Qatar                     2
Oman                      1
United Arab Emirates      1
dtype: int64

In [9]:
# 1. Define a list of maritime regions that should have 0 population
maritime_regions = ['Indian Ocean', 'Mediterranean Sea']

# 2. Set the population exposure to 0 ONLY for these maritime rows
is_maritime = df['COUNTRY'].isin(maritime_regions)
df.loc[is_maritime, 'POPULATION_EXPOSURE'] = df.loc[is_maritime, 'POPULATION_EXPOSURE'].fillna(0)

# 3. For all other rows (actual land countries), fill using the local region's median
df['POPULATION_EXPOSURE'] = df.groupby(['COUNTRY', 'ADMIN1'])['POPULATION_EXPOSURE'].transform(lambda x: x.fillna(x.median()))

# 4. Final safety fallback: If an entire country is missing numbers, fill with the overall country median
df['POPULATION_EXPOSURE'] = df.groupby('COUNTRY')['POPULATION_EXPOSURE'].transform(lambda x: x.fillna(x.median()))



c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [10]:
# Group by country using the exact column name
missing_by_country = df[df['POPULATION_EXPOSURE'].isnull()].groupby('COUNTRY').size()

# Sort the results in descending order
missing_by_country = missing_by_country.sort_values(ascending=False)
missing_by_country


Series([], dtype: int64)

In [11]:
df.isnull()

,WEEK,REGION,COUNTRY,ADMIN1,EVENT_TYPE,SUB_EVENT_TYPE,EVENTS,FATALITIES,POPULATION_EXPOSURE,DISORDER_TYPE,ID,CENTROID_LATITUDE,CENTROID_LONGITUDE
0,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
155290,False,False,False,False,False,False,False,False,False,False,False,False,False
155291,False,False,False,False,False,False,False,False,False,False,False,False,False
155292,False,False,False,False,False,False,False,False,False,False,False,False,False
155293,False,False,False,False,False,False,False,False,False,False,False,False,False


In [12]:
df.isnull().sum()

WEEK                   0
REGION                 0
COUNTRY                0
ADMIN1                 0
EVENT_TYPE             0
SUB_EVENT_TYPE         0
EVENTS                 0
FATALITIES             0
POPULATION_EXPOSURE    0
DISORDER_TYPE          0
ID                     0
CENTROID_LATITUDE      0
CENTROID_LONGITUDE     0
dtype: int64

In [13]:
df.isna().any().any()


np.False_

In [14]:
# loading data into cleaned csv file
# Saves the cleaned data to a new CSV file without adding an extra index column
df.to_csv('cleaned_conflict_data.csv', index=False)
